# LLM-jp-Moshi: Japanese Full-duplex Spoken Dialogue Model

このノートブックでは、[LLM-jp-Moshi-v1](https://huggingface.co/llm-jp/llm-jp-moshi-v1) をGoogle Colab上で動かします。

LLM-jp-Moshi-v1は、日本語のfull-duplex（全二重）音声対話システムです。英語の7Bパラメータ音声対話モデル [Moshi](https://arxiv.org/abs/2410.00037) をベースに、日本語音声対話データで追加学習して構築されています。

## 必要条件

- **GPU**: 24GB以上のVRAMが必要です（**A100推奨**）
  - Colab無料版のT4 (16GB) では不足する可能性があります
  - 「ランタイム」→「ランタイムのタイプを変更」→ A100 を選択してください
- **音声デバイス**: エコー防止のため、イヤホン・ヘッドホンの使用を推奨します

> **注意**: このモデルは試作段階であり、応答が不自然な場合があります。

## 1. GPU確認

まず、利用可能なGPUとVRAMを確認します。24GB以上のVRAMが必要です。

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)

# VRAM容量を確認
import torch
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_mem / (1024**3)
    print(f"\nGPU: {gpu_name}")
    print(f"VRAM: {vram_gb:.1f} GB")
    if vram_gb < 24:
        print(f"\n⚠️ 警告: VRAMが{vram_gb:.1f}GBしかありません。24GB以上が推奨されます。")
        print("「ランタイム」→「ランタイムのタイプを変更」からA100 GPUを選択してください。")
    else:
        print("\n✅ VRAMは十分です。")
else:
    print("❌ GPUが利用できません。ランタイムの設定を確認してください。")

## 2. パッケージのインストール

Moshi PyTorch実装とトンネリングツールをインストールします。

In [ ]:
!pip install "moshi<=0.2.2"

## 3. サーバーの起動

LLM-jp-Moshi-v1のサーバーをバックグラウンドで起動します。

初回実行時はHuggingFace Hubからモデル（約14GB）がダウンロードされるため、数分かかります。

In [ ]:
import subprocess
import time

# サーバーをバックグラウンドで起動
server_process = subprocess.Popen(
    ['python', '-m', 'moshi.server', '--hf-repo', 'llm-jp/llm-jp-moshi-v1'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

print("サーバーを起動中...（モデルのダウンロード・ロードに数分かかることがあります）")
print("ログを監視しています...\n")

# サーバーが起動するまでログを表示
import select
import os
import fcntl

# ノンブロッキング読み取りの設定
fd = server_process.stdout.fileno()
fl = fcntl.fcntl(fd, fcntl.F_GETFL)
fcntl.fcntl(fd, fcntl.F_SETFL, fl | os.O_NONBLOCK)

server_ready = False
start_time = time.time()
timeout = 600  # 10分タイムアウト

while time.time() - start_time < timeout:
    try:
        line = server_process.stdout.readline()
        if line:
            print(line, end='')
            # サーバー起動完了の判定（URLやportの表示を検出）
            if '8998' in line or 'http' in line.lower() or 'listening' in line.lower() or 'started' in line.lower():
                server_ready = True
                # もう少しログを読む
                time.sleep(2)
                try:
                    while True:
                        extra = server_process.stdout.readline()
                        if not extra:
                            break
                        print(extra, end='')
                except:
                    pass
                break
    except (IOError, TypeError):
        pass
    
    # プロセスが終了していないか確認
    if server_process.poll() is not None:
        remaining = server_process.stdout.read()
        if remaining:
            print(remaining)
        print(f"\n❌ サーバーが終了しました（終了コード: {server_process.returncode}）")
        break
    
    time.sleep(1)

if server_ready:
    print("\n✅ サーバーが起動しました！次のセルでトンネルを開始してください。")
elif time.time() - start_time >= timeout:
    print("\n⚠️ タイムアウトしました。サーバーの起動に時間がかかっています。")
    print("次のセルを実行してトンネルを試してみてください。")

## 4. 公開URLの作成（トンネリング）

Google Colabのサーバーに外部からアクセスするためのトンネルを作成します。

表示されたURLをブラウザで開くと、Moshiの対話UIにアクセスできます。

### 方法A: localtunnel を使用（推奨）

In [ ]:
!npm install -g localtunnel 2>/dev/null && npx localtunnel --port 8998

### 方法B: Colab のサーバープロキシを使用

localtunnelがうまくいかない場合は、以下を試してください。

> **注意**: WebSocketの通信がプロキシ経由でうまくいかない場合があります。

In [ ]:
from google.colab import output
output.serve_kernel_port_as_window(8998)

### 方法C: ngrok を使用

[ngrok](https://ngrok.com/) のアカウントがある場合は、こちらの方法も使えます。

`YOUR_NGROK_AUTHTOKEN` を自分のトークンに置き換えてください。

In [ ]:
# ngrokを使用する場合はコメントを外して実行してください
# !pip install pyngrok
# from pyngrok import ngrok
# ngrok.set_auth_token("YOUR_NGROK_AUTHTOKEN")
# public_url = ngrok.connect(8998)
# print(f"\n🌐 公開URL: {public_url}")
# print("このURLをブラウザで開いて対話してください。")

## 5. サーバーの停止

対話を終了する場合は、以下のセルを実行してサーバーを停止してください。

In [ ]:
try:
    server_process.terminate()
    server_process.wait(timeout=10)
    print("✅ サーバーを停止しました。")
except Exception as e:
    print(f"サーバーの停止中にエラーが発生しました: {e}")
    server_process.kill()
    print("サーバーを強制停止しました。")

## トラブルシューティング

### VRAM不足エラーが出る場合
- 「ランタイム」→「ランタイムのタイプを変更」→ **A100** を選択してください
- Colab無料版のT4 (16GB) では不足する可能性があります

### モデルのダウンロードが遅い場合
- HuggingFace Hubからのダウンロード（約14GB）には時間がかかります
- ダウンロード済みモデルはセッション中キャッシュされます

### 音声が聞こえない / マイクが使えない場合
- ブラウザのマイク・スピーカー権限を確認してください
- HTTPS経由でアクセスしていることを確認してください（localtunnel/ngrokは自動でHTTPS）
- エコー防止のため、イヤホン・ヘッドホンを使用してください

### WebSocket接続エラーが出る場合
- localtunnelのページで「Click to Continue」ボタンを押す必要がある場合があります
- 方法Cのngrokを試してみてください（より安定したトンネリングが可能）

## 参考リンク

- [LLM-jp-Moshi デモページ](https://llm-jp.github.io/llm-jp-moshi/)
- [HuggingFace Model](https://huggingface.co/llm-jp/llm-jp-moshi-v1)
- [Moshi オリジナルリポジトリ](https://github.com/kyutai-labs/moshi)
- [論文 (IWSDS 2026)](https://aclanthology.org/2026.iwsds-1.10/)